Notebook 6 — Frozen Held-Out Validation

This is the first real predictive test of the empirical law. It uses the validated variance_curves.csv, derives \(\tau_{BP}\) from scratch, fits only on \(n=8,10,12\), freezes the resulting parameters, and predicts \(n=14,16\) without refitting.

No new quantum simulation is required.


### Cell 1 — Setup


In [ ]:
# ============================================================
# NOTEBOOK 6 — FROZEN HELD-OUT VALIDATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

INPUT_FILE = "variance_curves.csv"

THRESHOLD = 1e-2

# STRICT DATA SPLIT
TRAIN_N = [8, 10, 12]
HELDOUT_N = [14, 16]

OUTPUT_DIR = "notebook6_frozen_validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 80)
print("NOTEBOOK 6 — FROZEN HELD-OUT VALIDATION")
print("=" * 80)
print(f"Input file:   {INPUT_FILE}")
print(f"Threshold:    {THRESHOLD}")
print(f"Training n:   {TRAIN_N}")
print(f"Held-out n:   {HELDOUT_N}")
print("Held-out refitting: NO")
print("Quantum simulation: NONE")
print("=" * 80)

### Cell 2 — Load raw data


In [ ]:
# ============================================================
# CELL 2 — LOAD RAW VARIANCE DATA
# ============================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"CRITICAL: {INPUT_FILE} not found."
    )

raw_df = pd.read_csv(INPUT_FILE)

required_columns = [
    "n",
    "k",
    "depth",
    "var_pooled",
]

missing = [
    c for c in required_columns
    if c not in raw_df.columns
]

if missing:
    raise ValueError(
        f"CRITICAL: Missing columns: {missing}"
    )

for c in required_columns:
    raw_df[c] = pd.to_numeric(
        raw_df[c],
        errors="coerce"
    )

if raw_df[required_columns].isna().any().any():
    raise ValueError(
        "CRITICAL: Missing/non-numeric values found."
    )

print(f"Rows: {len(raw_df):,}")
print(
    "n values:",
    sorted(raw_df["n"].astype(int).unique().tolist())
)
print(
    "k values:",
    sorted(raw_df["k"].astype(int).unique().tolist())
)
print(
    "Depth range:",
    int(raw_df["depth"].min()),
    "to",
    int(raw_df["depth"].max())
)

### Cell 3 — Reconstruct \(\tau_{BP}\)


In [ ]:
# ============================================================
# CELL 3 — RECONSTRUCT TAU_BP DIRECTLY FROM RAW CURVES
# ============================================================

MAX_DEPTH = int(raw_df["depth"].max())

tau_records = []

for (n, k), group in raw_df.groupby(["n", "k"]):

    group = group.sort_values("depth")

    crossed = group[
        group["var_pooled"] < THRESHOLD
    ]

    if len(crossed) > 0:
        tau = int(crossed.iloc[0]["depth"])
        censored = False
    else:
        tau = MAX_DEPTH + 1
        censored = True

    tau_records.append({
        "n": int(n),
        "k": int(k),
        "nk": int(n * k),
        "tau_BP": tau,
        "censored": censored,
    })

tau_df = pd.DataFrame(tau_records)

tau_df = tau_df.sort_values(
    ["n", "k"]
).reset_index(drop=True)

print("=" * 80)
print("RECONSTRUCTED TAU_BP")
print("=" * 80)

display(tau_df)

print(
    f"\nConfigurations: {len(tau_df)}"
)
print(
    f"Censored: {int(tau_df['censored'].sum())}"
)

### Cell 4 — Enforce the frozen split


In [ ]:
# ============================================================
# CELL 4 — TRAIN / HELD-OUT SPLIT AUDIT
# ============================================================

train_df = tau_df[
    tau_df["n"].isin(TRAIN_N)
].copy()

heldout_df = tau_df[
    tau_df["n"].isin(HELDOUT_N)
].copy()

if len(train_df) == 0:
    raise RuntimeError(
        "CRITICAL: Training set is empty."
    )

if len(heldout_df) == 0:
    raise RuntimeError(
        "CRITICAL: Held-out set is empty."
    )

# Training data must be exact observations for this OLS analysis.
if train_df["censored"].any():
    raise RuntimeError(
        "CRITICAL: Training set contains censored observations."
    )

print("=" * 80)
print("FROZEN DATA PARTITION")
print("=" * 80)

print("Training n:", sorted(train_df["n"].unique().tolist()))
print("Held-out n:", sorted(heldout_df["n"].unique().tolist()))

print(
    f"Training configurations: {len(train_df)}"
)
print(
    f"Held-out configurations: {len(heldout_df)}"
)

print("\nTraining:")
display(
    train_df[
        ["n", "k", "nk", "tau_BP", "censored"]
    ]
)

print("\nHeld-out:")
display(
    heldout_df[
        ["n", "k", "nk", "tau_BP", "censored"]
    ]
)

overlap = (
    set(train_df["n"])
    &
    set(heldout_df["n"])
)

if overlap:
    raise RuntimeError(
        f"CRITICAL: Data leakage detected: {overlap}"
    )

### Cell 5 — Fit the empirical law ONCE


In [ ]:
# ============================================================
# CELL 5 — FIT FROZEN PRODUCT MODEL
# ============================================================

train_fit = train_df.copy()

train_fit["log_tau"] = np.log(
    train_fit["tau_BP"].astype(float)
)

train_fit["log_nk"] = np.log(
    train_fit["nk"].astype(float)
)

X_train = sm.add_constant(
    train_fit["log_nk"]
)

frozen_model = sm.OLS(
    train_fit["log_tau"],
    X_train
).fit()

log_A = float(
    frozen_model.params["const"]
)

slope = float(
    frozen_model.params["log_nk"]
)

A_frozen = float(
    np.exp(log_A)
)

c_frozen = float(
    -slope
)

print("=" * 80)
print("FROZEN TRAINING MODEL")
print("=" * 80)

print(f"A = {A_frozen:.10f}")
print(f"c = {c_frozen:.10f}")

print(
    "\nFrozen empirical law:"
)

print(
    f"tau_BP = {A_frozen:.10f} * "
    f"(n*k)^(-{c_frozen:.10f})"
)

print(
    "\nFrom this point onward A and c MUST NOT change."
)

### Cell 6 — Training fit diagnostics


In [ ]:
# ============================================================
# CELL 6 — TRAINING MODEL DIAGNOSTICS
# ============================================================

train_fit["tau_pred_train"] = (
    A_frozen
    * train_fit["nk"].astype(float)
    .pow(-c_frozen)
)

train_fit["residual_train"] = (
    train_fit["tau_BP"]
    - train_fit["tau_pred_train"]
)

train_mae = mean_absolute_error(
    train_fit["tau_BP"],
    train_fit["tau_pred_train"]
)

train_rmse = np.sqrt(
    mean_squared_error(
        train_fit["tau_BP"],
        train_fit["tau_pred_train"]
    )
)

train_r2 = r2_score(
    train_fit["tau_BP"],
    train_fit["tau_pred_train"]
)

print("=" * 80)
print("TRAINING FIT")
print("=" * 80)

print(f"MAE:  {train_mae:.6f}")
print(f"RMSE: {train_rmse:.6f}")
print(f"R²:   {train_r2:.6f}")

display(
    train_fit[
        [
            "n",
            "k",
            "nk",
            "tau_BP",
            "tau_pred_train",
            "residual_train",
        ]
    ]
)

### Cell 7 — Predict held-out systems WITHOUT refitting


In [ ]:
# ============================================================
# CELL 7 — FROZEN HELD-OUT PREDICTIONS
# ============================================================

validation_df = heldout_df.copy()

validation_df["tau_pred"] = (
    A_frozen
    * validation_df["nk"].astype(float)
    .pow(-c_frozen)
)

# Integer-depth diagnostic
validation_df["tau_pred_rounded"] = np.rint(
    validation_df["tau_pred"]
).astype(int)

validation_df["residual"] = (
    validation_df["tau_BP"]
    - validation_df["tau_pred"]
)

validation_df["absolute_error"] = (
    validation_df["residual"].abs()
)

validation_df["percentage_error"] = (
    validation_df["absolute_error"]
    / validation_df["tau_BP"].abs()
    * 100
)

validation_df["rounded_residual"] = (
    validation_df["tau_BP"]
    - validation_df["tau_pred_rounded"]
)

validation_df["rounded_absolute_error"] = (
    validation_df["rounded_residual"].abs()
)

print("=" * 80)
print("FROZEN HELD-OUT PREDICTIONS")
print("=" * 80)

display(
    validation_df[
        [
            "n",
            "k",
            "nk",
            "tau_BP",
            "tau_pred",
            "tau_pred_rounded",
            "residual",
            "absolute_error",
            "percentage_error",
        ]
    ]
)

### Cell 8 — Evaluation metrics


In [ ]:
# ============================================================
# CELL 8 — HELD-OUT METRICS
# ============================================================

def evaluate_subset(
    df,
    prediction_column
):
    y_true = df["tau_BP"].astype(float).to_numpy()
    y_pred = df[prediction_column].astype(float).to_numpy()

    residual = y_true - y_pred

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mape = np.mean(
        np.abs(residual / y_true)
    ) * 100

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "N": len(df),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "MAPE_percent": float(mape),
        "R2": float(r2),
        "mean_signed_error": float(
            residual.mean()
        ),
        "within_1_layer_percent": float(
            100 * np.mean(
                np.abs(residual) <= 1
            )
        ),
        "within_2_layers_percent": float(
            100 * np.mean(
                np.abs(residual) <= 2
            )
        ),
    }


metrics_rows = []

for n_val in HELDOUT_N:

    subset = validation_df[
        validation_df["n"] == n_val
    ]

    if len(subset) > 0:

        metrics = evaluate_subset(
            subset,
            "tau_pred"
        )

        metrics["held_out_n"] = n_val

        metrics_rows.append(
            metrics
        )

combined = evaluate_subset(
    validation_df,
    "tau_pred"
)

combined["held_out_n"] = "14+16"

metrics_rows.append(
    combined
)

validation_metrics = pd.DataFrame(
    metrics_rows
)

print("=" * 80)
print("FROZEN HELD-OUT PERFORMANCE")
print("=" * 80)

display(
    validation_metrics[
        [
            "held_out_n",
            "N",
            "MAE",
            "RMSE",
            "MAPE_percent",
            "R2",
            "mean_signed_error",
            "within_1_layer_percent",
            "within_2_layers_percent",
        ]
    ]
)

### Cell 9 — Integer-depth prediction metrics


In [ ]:
# ============================================================
# CELL 9 — INTEGER DEPTH DIAGNOSTIC
# ============================================================

rounded_rows = []

for n_val in HELDOUT_N:

    subset = validation_df[
        validation_df["n"] == n_val
    ]

    if len(subset) > 0:

        metrics = evaluate_subset(
            subset,
            "tau_pred_rounded"
        )

        metrics["held_out_n"] = n_val

        rounded_rows.append(
            metrics
        )

combined_rounded = evaluate_subset(
    validation_df,
    "tau_pred_rounded"
)

combined_rounded["held_out_n"] = "14+16"

rounded_rows.append(
    combined_rounded
)

rounded_metrics = pd.DataFrame(
    rounded_rows
)

print("=" * 80)
print("INTEGER-DEPTH HELD-OUT PERFORMANCE")
print("=" * 80)

display(
    rounded_metrics[
        [
            "held_out_n",
            "N",
            "MAE",
            "RMSE",
            "MAPE_percent",
            "R2",
            "mean_signed_error",
            "within_1_layer_percent",
            "within_2_layers_percent",
        ]
    ]
)

### Cell 10 — Bias / residual audit


In [ ]:
# ============================================================
# CELL 10 — RESIDUAL AUDIT
# ============================================================

bias_rows = []

for n_val in HELDOUT_N:

    subset = validation_df[
        validation_df["n"] == n_val
    ]

    bias_rows.append({
        "n": n_val,
        "mean_residual":
            subset["residual"].mean(),
        "median_residual":
            subset["residual"].median(),
        "MAE":
            subset["absolute_error"].mean(),
        "max_absolute_error":
            subset["absolute_error"].max(),
    })

bias_df = pd.DataFrame(
    bias_rows
)

print("=" * 80)
print("HELD-OUT BIAS AUDIT")
print("=" * 80)

display(
    bias_df
)

print(
    "Residual = observed tau_BP - predicted tau_BP"
)

### Cell 11 — Observed vs predicted


In [ ]:
# ============================================================
# CELL 11 — OBSERVED VS PREDICTED
# ============================================================

plt.figure(
    figsize=(9, 7)
)

plt.scatter(
    validation_df["tau_BP"],
    validation_df["tau_pred"],
    s=70,
    label="Held-out observations"
)

values = np.concatenate([
    validation_df["tau_BP"].to_numpy(),
    validation_df["tau_pred"].to_numpy(),
])

lo = np.floor(values.min())
hi = np.ceil(values.max())

plt.plot(
    [lo, hi],
    [lo, hi],
    linestyle="--",
    linewidth=2,
    label="Perfect prediction"
)

plt.xlabel(
    "Observed tau_BP"
)

plt.ylabel(
    "Frozen-model predicted tau_BP"
)

plt.title(
    "Frozen Out-of-Domain Prediction: n=14,16"
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "observed_vs_predicted.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()

### Cell 12 — Residual plot


In [ ]:
# ============================================================
# CELL 12 — HELD-OUT RESIDUALS
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.scatter(
    validation_df["nk"],
    validation_df["residual"],
    s=70
)

plt.xscale("log")

plt.xlabel(
    "n × k"
)

plt.ylabel(
    "Observed − Predicted tau_BP"
)

plt.title(
    "Frozen Held-Out Residuals"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "heldout_residuals.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()

### Cell 13 — Leakage test


In [ ]:
# ============================================================
# CELL 13 — FINAL ANTI-LEAKAGE CHECK
# ============================================================

training_n = set(
    train_fit["n"].astype(int)
)

heldout_n = set(
    validation_df["n"].astype(int)
)

print("=" * 80)
print("ANTI-LEAKAGE AUDIT")
print("=" * 80)

print(
    "Training n:",
    sorted(training_n)
)

print(
    "Held-out n:",
    sorted(heldout_n)
)

print(
    "Overlap:",
    sorted(training_n & heldout_n)
)

if training_n & heldout_n:
    raise RuntimeError(
        "CRITICAL: Data leakage detected."
    )

print(
    "\nPASS: No held-out n value entered the fit."
)

print(
    "PASS: Held-out predictions used frozen A and c only."
)

### Cell 14 — Final summary


In [ ]:
# ============================================================
# CELL 14 — FINAL SCIENTIFIC SUMMARY
# ============================================================

combined_row = validation_metrics[
    validation_metrics["held_out_n"] == "14+16"
].iloc[0]

print("\n" + "=" * 90)
print("CHECKPOINT 6 — FINAL SUMMARY")
print("=" * 90)

print("\nFROZEN MODEL")
print("-" * 90)

print(
    f"A = {A_frozen:.10f}"
)

print(
    f"c = {c_frozen:.10f}"
)

print(
    f"Training domain = n={TRAIN_N}"
)

print(
    f"Held-out domain = n={HELDOUT_N}"
)

print("\nCOMBINED HELD-OUT PERFORMANCE")
print("-" * 90)

print(
    f"N = {int(combined_row['N'])}"
)

print(
    f"MAE = {combined_row['MAE']:.6f}"
)

print(
    f"RMSE = {combined_row['RMSE']:.6f}"
)

print(
    f"MAPE = {combined_row['MAPE_percent']:.4f}%"
)

print(
    f"R² = {combined_row['R2']:.6f}"
)

print(
    f"Within ±1 layer = "
    f"{combined_row['within_1_layer_percent']:.2f}%"
)

print(
    f"Within ±2 layers = "
    f"{combined_row['within_2_layers_percent']:.2f}%"
)

print("\nSCIENTIFIC INTERPRETATION")
print("-" * 90)

print(
    "The held-out result tests extrapolation to unseen system sizes."
)

print(
    "The held-out n=14 and n=16 observations were not used to "
    "estimate A or c."
)

print(
    "No claim of universal validity should be made from this "
    "single finite validation domain."
)

### Cell 15 — Save publication outputs


In [ ]:
# ============================================================
# CELL 15 — SAVE CHECKPOINT 6 OUTPUTS
# ============================================================

tau_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "reconstructed_tau_all.csv"
    ),
    index=False
)

train_fit.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "frozen_training_data.csv"
    ),
    index=False
)

validation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "frozen_heldout_predictions.csv"
    ),
    index=False
)

validation_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "frozen_validation_metrics.csv"
    ),
    index=False
)

rounded_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "frozen_validation_metrics_rounded.csv"
    ),
    index=False
)

bias_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "heldout_bias_audit.csv"
    ),
    index=False
)

summary = {
    "checkpoint": 6,
    "input_file": INPUT_FILE,
    "threshold": THRESHOLD,

    "training_n": TRAIN_N,
    "heldout_n": HELDOUT_N,

    "A_frozen": A_frozen,
    "c_frozen": c_frozen,

    "training_observations": int(
        len(train_fit)
    ),

    "heldout_observations": int(
        len(validation_df)
    ),

    "heldout_MAE": float(
        combined_row["MAE"]
    ),

    "heldout_RMSE": float(
        combined_row["RMSE"]
    ),

    "heldout_MAPE_percent": float(
        combined_row["MAPE_percent"]
    ),

    "heldout_R2": float(
        combined_row["R2"]
    ),

    "heldout_within_1_layer_percent":
        float(
            combined_row[
                "within_1_layer_percent"
            ]
        ),

    "heldout_within_2_layers_percent":
        float(
            combined_row[
                "within_2_layers_percent"
            ]
        ),

    "data_leakage": False,
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "checkpoint6_summary.json"
    ),
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print("=" * 80)
print("CHECKPOINT 6 DELIVERABLES")
print("=" * 80)

for fname in sorted(
    os.listdir(OUTPUT_DIR)
):
    print(
        os.path.join(
            OUTPUT_DIR,
            fname
        )
    )

print("\nSTATUS: COMPLETE")

The critical output to send me is Cell 14, especially the frozen \(A,c\), combined MAE/RMSE/MAPE/\(R^2\), and the individual predictions in Cell 7.
